# Day 2 — Data Cleaning & Database Loading

This notebook demonstrates the complete Day 2 pipeline:
1. Clean NAV history (parse dates, forward-fill weekends/holidays, remove duplicates)
2. Clean investor transactions (standardize types, validate amounts & KYC)
3. Clean scheme performance (coerce numerics, validate ranges)
4. Verify the SQLite database tables loaded by `db_loader.py`
5. Execute the 10 analytical queries from `sql/queries.sql`

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

BASE_DIR       = Path().resolve().parent
RAW_DIR        = BASE_DIR / "data" / "raw"
PROCESSED_DIR  = BASE_DIR / "data" / "processed"
DB_PATH        = BASE_DIR / "data" / "db" / "bluestock_mf.db"
SQL_DIR        = BASE_DIR / "sql"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw data   : {RAW_DIR}")
print(f"Processed  : {PROCESSED_DIR}")
print(f"Database   : {DB_PATH}")

## 1. Clean NAV History

- Parse dates to datetime
- Sort by `amfi_code` then `date`
- Reindex to full business-day range per fund & forward-fill gaps
- Remove duplicates and validate `nav > 0`

In [ ]:
nav_raw = pd.read_csv(RAW_DIR / "02_nav_history.csv")
print(f"Raw NAV records: {len(nav_raw):,}")
print(f"Duplicates     : {nav_raw.duplicated().sum()}")
print(f"Null NAVs      : {nav_raw['nav'].isnull().sum()}")

# Remove duplicates
nav = nav_raw.drop_duplicates()

# Parse dates
nav['date'] = pd.to_datetime(nav['date'], format='mixed', dayfirst=True)

# Sort
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# Validate nav > 0
invalid = nav[nav['nav'] <= 0]
if not invalid.empty:
    print(f"⚠️ Dropping {len(invalid)} rows with NAV <= 0")
    nav = nav[nav['nav'] > 0]

# Reindex to business days and forward-fill
def reindex_fund(group):
    amfi = group['amfi_code'].iloc[0]
    group = group.set_index('date')
    group = group[~group.index.duplicated(keep='last')]
    if group.empty:
        return group
    idx = pd.bdate_range(start=group.index.min(), end=group.index.max())
    group = group.reindex(idx)
    group['amfi_code'] = amfi
    group['nav'] = group['nav'].ffill()
    return group.rename_axis('date').reset_index()

nav_clean = nav.groupby('amfi_code', group_keys=False).apply(reindex_fund).dropna(subset=['nav'])
nav_clean = nav_clean.sort_values(['amfi_code', 'date']).reset_index(drop=True)
nav_clean = nav_clean[['amfi_code', 'date', 'nav']]

nav_clean.to_csv(PROCESSED_DIR / "clean_nav.csv", index=False)
print(f"\n✅ Cleaned NAV: {len(nav_raw):,} → {len(nav_clean):,} rows")
print(f"   Forward-filled {len(nav_clean) - len(nav):,} weekend/holiday gaps")
display(nav_clean.head(10))

## 2. Clean Investor Transactions

- Parse `transaction_date`
- Standardize `transaction_type` to `SIP`, `Lumpsum`, `Redemption`
- Validate `amount_inr > 0` and `kyc_status ∈ {Verified, Pending}`

In [ ]:
tx_raw = pd.read_csv(RAW_DIR / "08_investor_transactions.csv")
print(f"Raw transactions: {len(tx_raw):,}")
print(f"Transaction types: {tx_raw['transaction_type'].unique()}")

tx = tx_raw.copy()

# Parse dates
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'], format='mixed', dayfirst=True)

# Standardize transaction_type
tx['transaction_type'] = tx['transaction_type'].str.strip().str.title()
type_map = {'Sip': 'SIP', 'Lumpsum': 'Lumpsum', 'Redemption': 'Redemption'}
tx['transaction_type'] = tx['transaction_type'].map(type_map).fillna(tx['transaction_type'])
tx = tx[tx['transaction_type'].isin(['SIP', 'Lumpsum', 'Redemption'])]

# Validate amount > 0
invalid_amt = tx[tx['amount_inr'] <= 0]
if not invalid_amt.empty:
    print(f"⚠️ Dropping {len(invalid_amt)} rows with amount <= 0")
    tx = tx[tx['amount_inr'] > 0]

# Validate KYC
tx['kyc_status'] = tx['kyc_status'].str.strip().str.title()
tx = tx[tx['kyc_status'].isin(['Verified', 'Pending'])]

tx.to_csv(PROCESSED_DIR / "clean_transactions.csv", index=False)
print(f"\n✅ Cleaned transactions: {len(tx_raw):,} → {len(tx):,} rows")
print(f"\nTransaction type distribution:")
display(tx['transaction_type'].value_counts())
print(f"\nKYC status distribution:")
display(tx['kyc_status'].value_counts())

## 3. Clean Scheme Performance

- Coerce return/ratio columns to numeric
- Flag negative Sharpe ratios
- Validate expense ratio range (0.1%–2.5%)

In [ ]:
perf_raw = pd.read_csv(RAW_DIR / "07_scheme_performance.csv")
print(f"Raw performance records: {len(perf_raw)}")

perf = perf_raw.copy()

numeric_cols = [
    'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
    'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio',
    'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct'
]

for col in numeric_cols:
    if col in perf.columns:
        perf[col] = pd.to_numeric(perf[col], errors='coerce')

# Flag negative Sharpe
neg_sharpe = perf[perf['sharpe_ratio'] < 0]
if not neg_sharpe.empty:
    print(f"⚠️ {len(neg_sharpe)} schemes with negative Sharpe ratio")

# Validate expense ratio
perf['expense_ratio_pct'] = pd.to_numeric(perf['expense_ratio_pct'], errors='coerce')
anomalies = perf[(perf['expense_ratio_pct'] < 0.1) | (perf['expense_ratio_pct'] > 2.5)]
if not anomalies.empty:
    print(f"⚠️ {len(anomalies)} schemes with expense_ratio outside 0.1%–2.5%")

perf.to_csv(PROCESSED_DIR / "clean_performance.csv", index=False)
print(f"\n✅ Cleaned performance: {len(perf)} rows")
display(perf[['scheme_name', 'sharpe_ratio', 'sortino_ratio', 'return_3yr_pct', 'risk_grade']].head(10))

## 4. Verify SQLite Database Tables

Connect to `bluestock_mf.db` and verify all expected tables are loaded with correct row counts.

In [ ]:
if DB_PATH.exists():
    conn = sqlite3.connect(str(DB_PATH))
    cursor = conn.cursor()

    # List all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = [row[0] for row in cursor.fetchall()]
    print(f"Tables in bluestock_mf.db ({len(tables)}):")

    table_info = []
    for tbl in tables:
        cursor.execute(f"SELECT COUNT(*) FROM [{tbl}]")
        count = cursor.fetchone()[0]
        table_info.append({"Table": tbl, "Rows": f"{count:,}"})
        print(f"  📋 {tbl:30s} {count:>8,} rows")

    display(pd.DataFrame(table_info))
    conn.close()
else:
    print("⚠️ Database not found. Run `python scripts/etl_pipeline.py` first.")

## 5. Execute Analytical Queries from `queries.sql`

Run the 10 pre-defined queries and display results inline.

In [ ]:
import re

if DB_PATH.exists():
    conn = sqlite3.connect(str(DB_PATH))
    queries_file = SQL_DIR / "queries.sql"

    if queries_file.exists():
        sql_text = queries_file.read_text(encoding="utf-8")

        # Split on comment lines that start with '--' and contain a query title
        # Each query block = comment header + SQL
        blocks = re.split(r'(?=^-- \d+\.)', sql_text, flags=re.MULTILINE)
        blocks = [b.strip() for b in blocks if b.strip()]

        for i, block in enumerate(blocks, 1):
            lines = block.strip().split('\n')
            # Extract title from first comment line
            title = lines[0].lstrip('- ').strip() if lines[0].startswith('--') else f"Query {i}"
            # Extract SQL (non-comment lines)
            sql = '\n'.join(l for l in lines if not l.strip().startswith('--')).strip()
            if not sql:
                continue

            print(f"\n{'=' * 70}")
            print(f"📊 {title}")
            print(f"{'=' * 70}")
            try:
                result = pd.read_sql_query(sql, conn)
                display(result.head(10))
                print(f"   ({len(result)} rows returned)")
            except Exception as e:
                print(f"   ❌ Error: {e}")
    else:
        print("⚠️ queries.sql not found")

    conn.close()
else:
    print("⚠️ Database not found. Run `python scripts/etl_pipeline.py` first.")

---
*Day 2 Data Cleaning & Database Loading complete. All 3 core datasets cleaned, SQLite DB verified, and analytical queries executed.*